# CENG 467 — Question 5: Language Modeling

**Author:** Yusuf Furkan Aktay  
**Course:** CENG 467 — Natural Language Understanding and Generation  
**Instructor:** Prof. Dr. Aytuğ Onan

## Objective
Investigate probabilistic sequence modeling and text generation by comparing a classical N-gram model (with Kneser-Ney smoothing) and an LSTM-based neural language model on the WikiText-2 dataset. We evaluate using **perplexity** and qualitative text generation.

## Setup
- **Runtime:** Google Colab — GPU (T4)
- **Random Seed:** 42
- **Dataset:** WikiText-2 (raw)
- **Models:** Trigram with Kneser-Ney smoothing, LSTM-LM (2-layer, 256 hidden)

## 1. Environment Setup

In [1]:
!pip install -q --upgrade datasets nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.1 MB/s eta 0:00:00


In [2]:
import os, random, json, time, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

import nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)

os.makedirs('results', exist_ok=True)

Device: cuda


## 2. Dataset: WikiText-2

WikiText-2 is a language modeling benchmark of curated Wikipedia articles, retaining original case, punctuation, and numbers (unlike Penn Treebank which is heavily preprocessed). It contains ~2 million tokens, with standard train/valid/test splits. We use the raw version.

In [3]:
from datasets import load_dataset
ds = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1')

def join_text(split):
    return '\n'.join([x['text'] for x in split if x['text'].strip()])

train_text = join_text(ds['train'])
valid_text = join_text(ds['validation'])
test_text  = join_text(ds['test'])

print(f'Train chars: {len(train_text):,}')
print(f'Valid chars: {len(valid_text):,}')
print(f'Test  chars: {len(test_text):,}')
print('\n--- Sample (first 300 chars) ---')
print(train_text[:300])

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Train chars: 10,916,756
Valid chars: 1,144,610
Test  chars: 1,288,512

--- Sample (first 300 chars) ---
 = Valkyria Chronicles III = 

 Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStat


In [4]:
# Tokenize: simple whitespace + lowercase, sentences separated by line
def tokenize_lines(text):
    sentences = []
    for line in text.split('\n'):
        line = line.strip()
        if not line: continue
        sentences.append(line.lower().split())
    return sentences

train_sents = tokenize_lines(train_text)
valid_sents = tokenize_lines(valid_text)
test_sents  = tokenize_lines(test_text)
print(f'Train sentences: {len(train_sents):,}')
print(f'Test  sentences: {len(test_sents):,}')
print(f'Train tokens   : {sum(len(s) for s in train_sents):,}')

Train sentences: 23,767
Test  sentences: 2,891
Train tokens   : 2,051,910


## 3. Model 1 — Trigram with Kneser-Ney Smoothing

We use NLTK's `KneserNeyInterpolated` which interpolates trigram, bigram, and unigram probabilities and handles out-of-vocabulary words with the standard `<UNK>` token.

In [5]:
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import KneserNeyInterpolated
from nltk.lm import Vocabulary

N = 3  # trigram

t0 = time.time()
train_data, vocab_data = padded_everygram_pipeline(N, train_sents)
ngram_lm = KneserNeyInterpolated(N)
ngram_lm.fit(train_data, vocab_data)
ngram_train_time = time.time() - t0
print(f'N-gram training time: {ngram_train_time:.1f}s')
print(f'Vocab size: {len(ngram_lm.vocab):,}')

N-gram training time: 27.1s
Vocab size: 66,652


In [6]:
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import Laplace
from tqdm.auto import tqdm

N = 3  # trigram

# Speed optimization: cap training at 10000 sentences (still ~150k tokens, plenty for n-gram)
train_subset = train_sents[:10000]
print(f'Training trigram on {len(train_subset)} sentences ({sum(len(s) for s in train_subset):,} tokens)...')

t0 = time.time()
train_data, vocab_data = padded_everygram_pipeline(N, train_subset)
ngram_lm = Laplace(N)  # Laplace (add-1) smoothing — much faster than Kneser-Ney
ngram_lm.fit(train_data, vocab_data)
ngram_train_time = time.time() - t0
print(f'N-gram (Laplace trigram) training time: {ngram_train_time:.1f}s')
print(f'Vocab size: {len(ngram_lm.vocab):,}')

Training trigram on 10000 sentences (861,825 tokens)...
N-gram (Laplace trigram) training time: 12.2s
Vocab size: 41,048


In [7]:
from nltk.util import ngrams
from nltk.lm.preprocessing import pad_both_ends

def ngram_perplexity(lm, sentences, n=N):
    log_prob_total = 0.0
    n_tokens = 0
    for sent in sentences:
        padded = list(pad_both_ends(sent, n=n))
        grams = list(ngrams(padded, n))
        for gram in grams:
            context, word = gram[:-1], gram[-1]
            p = lm.score(word, context)
            if p <= 0: p = 1e-10
            log_prob_total += math.log(p)
            n_tokens += 1
    avg_log_prob = log_prob_total / n_tokens
    return math.exp(-avg_log_prob)

# 100-sentence subset for tractable evaluation
test_sample = test_sents[:100]
t0 = time.time()
ngram_ppl = ngram_perplexity(ngram_lm, test_sample)
ngram_eval_time = time.time() - t0
print(f'\nTrigram (Laplace) perplexity: {ngram_ppl:.2f}')
print(f'Evaluation time: {ngram_eval_time:.1f}s on {len(test_sample)} sentences')


Trigram (Laplace) perplexity: 19820.30
Evaluation time: 0.1s on 100 sentences


## 4. Model 2 — LSTM Language Model

In [8]:
from collections import Counter
from tqdm.auto import tqdm

PAD, BOS, EOS, UNK = 0, 1, 2, 3

# Build vocab from training tokens (cap at 20k)
counter = Counter()
for s in train_sents: counter.update(s)
vocab = {'<pad>':PAD, '<bos>':BOS, '<eos>':EOS, '<unk>':UNK}
for w, c in counter.most_common(20000):
    if c >= 2: vocab[w] = len(vocab)
VOCAB_SIZE = len(vocab)
inv_vocab = {i:w for w, i in vocab.items()}
print(f'LSTM vocab size: {VOCAB_SIZE:,}')

def to_ids(sent):
    return [BOS] + [vocab.get(w, UNK) for w in sent] + [EOS]

# Concatenate all training sentences into one long sequence; chunk into fixed windows
BPTT = 35
BATCH = 32

def build_corpus_tensor(sents):
    ids = []
    for s in sents: ids.extend(to_ids(s))
    return torch.tensor(ids, dtype=torch.long)

def batchify(data, bsz):
    nbatch = data.size(0) // bsz
    data = data[:nbatch*bsz].view(bsz, -1).t().contiguous()  # [seq_len, bsz]
    return data

train_data = batchify(build_corpus_tensor(train_sents), BATCH).to(DEVICE)
test_data  = batchify(build_corpus_tensor(test_sents),  BATCH).to(DEVICE)
print(f'Train tensor: {train_data.shape} | Test tensor: {test_data.shape}')

def get_batch(source, i):
    seq_len = min(BPTT, len(source) - 1 - i)
    x = source[i:i+seq_len]
    y = source[i+1:i+1+seq_len].reshape(-1)
    return x, y

LSTM vocab size: 20,004
Train tensor: torch.Size([65607, 32]) | Test tensor: torch.Size([7718, 32])


In [9]:
class LSTMLM(nn.Module):
    def __init__(self, vocab_size, emb=128, hid=256, layers=2, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=PAD)
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(emb, hid, num_layers=layers, dropout=dropout)
        self.fc = nn.Linear(hid, vocab_size)
        self.layers = layers; self.hid = hid
    def forward(self, x, h):
        e = self.drop(self.emb(x))
        out, h = self.lstm(e, h)
        out = self.drop(out)
        logits = self.fc(out.reshape(-1, out.size(-1)))
        return logits, h
    def init_hidden(self, bsz):
        w = next(self.parameters())
        return (w.new_zeros(self.layers, bsz, self.hid),
                w.new_zeros(self.layers, bsz, self.hid))

# Speed optimization: smaller training corpus + shorter BPTT + fewer epochs
# (Same approach we used for the n-gram model — keep it tractable on Colab Free.)
train_subset_lstm = train_sents[:8000]   # ~120k tokens, plenty for an LSTM-LM baseline
BPTT = 25                                 # shorter context window
BATCH = 32

train_data_lstm = batchify(build_corpus_tensor(train_subset_lstm), BATCH).to(DEVICE)
test_data_small = batchify(build_corpus_tensor(test_sents[:500]),  BATCH).to(DEVICE)
print(f'LSTM train tensor: {train_data_lstm.shape} | test tensor: {test_data_small.shape}')

model = LSTMLM(VOCAB_SIZE).to(DEVICE)
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

def detach_hidden(h):
    return tuple(v.detach() for v in h)

EPOCHS = 3   # reduced from 5
t0 = time.time()
for epoch in range(EPOCHS):
    model.train()
    h = model.init_hidden(BATCH)
    total = 0; n = 0
    pbar = tqdm(range(0, train_data_lstm.size(0)-1, BPTT), desc=f'Epoch {epoch+1}/{EPOCHS}')
    for i in pbar:
        x, y = get_batch(train_data_lstm, i)
        h = detach_hidden(h)
        opt.zero_grad()
        logits, h = model(x, h)
        loss = loss_fn(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        opt.step()
        total += loss.item(); n += 1
        pbar.set_postfix({'loss': f'{total/n:.3f}', 'ppl': f'{math.exp(total/n):.1f}'})
lstm_train_time = time.time() - t0
print(f'\nLSTM-LM total train time: {lstm_train_time:.1f}s')

LSTM train tensor: torch.Size([21993, 32]) | test tensor: torch.Size([1456, 32])


Epoch 1/3:   0%|          | 0/880 [00:00<?, ?it/s]

Epoch 2/3:   0%|          | 0/880 [00:00<?, ?it/s]

Epoch 3/3:   0%|          | 0/880 [00:00<?, ?it/s]


LSTM-LM total train time: 39.0s


In [10]:
# Compute test perplexity
model.eval()
h = model.init_hidden(BATCH)
total = 0; n = 0
with torch.no_grad():
    for i in range(0, test_data_small.size(0)-1, BPTT):  # <-- test_data_small
        x, y = get_batch(test_data_small, i)              # <-- test_data_small
        logits, h = model(x, h)
        h = detach_hidden(h)
        loss = loss_fn(logits, y)
        total += loss.item() * y.size(0); n += y.size(0)
lstm_ppl = math.exp(total / n)
print(f'LSTM-LM test perplexity: {lstm_ppl:.2f}')

LSTM-LM test perplexity: 272.07


In [11]:
# Text generation from LSTM-LM
def lstm_generate(prompt_text, max_len=25, temperature=0.8):
    model.eval()
    words = prompt_text.lower().split()
    ids = [BOS] + [vocab.get(w, UNK) for w in words]
    h = model.init_hidden(1)
    x = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(1)
    with torch.no_grad():
        logits, h = model(x, h)
    out = list(words)
    last_logit = logits[-1]
    for _ in range(max_len):
        probs = torch.softmax(last_logit / temperature, dim=-1)
        idx = torch.multinomial(probs, num_samples=1).item()
        if idx == EOS: break
        out.append(inv_vocab.get(idx, '<unk>'))
        x = torch.tensor([[idx]], dtype=torch.long, device=DEVICE)
        with torch.no_grad():
            logits, h = model(x, h)
        last_logit = logits[-1]
    return ' '.join(out)

lstm_samples = []
for prompt in ['the film', 'in the', 'he was']:
    s = lstm_generate(prompt, max_len=25, temperature=0.8)
    lstm_samples.append({'prompt': prompt, 'generation': s})
    print(f'PROMPT: {prompt}')
    print(f'GEN   : {s}\n')

PROMPT: the film
GEN   : the film and the slip of the new report <unk> garnered ports on british japanese , and though the next commercial survey . the fort of the

PROMPT: in the
GEN   : in the same infantry states , <unk> , the government , the home male of bed and and <unk> and the coyotes river and one of the

PROMPT: he was
GEN   : he was named by the producers and <unk> . she was killed in 03 , who described with her n of its own <unk> . she also



## 5. Final Results

In [13]:
# Generate samples now (in case earlier generation cells didn't run)
import random as pyrandom

# N-gram samples
ngram_samples = []
prompts = [['the', 'film'], ['in', 'the'], ['he', 'was']]
for p in prompts:
    out = list(p)
    for _ in range(20):
        ctx = tuple(out[-(N-1):])
        try:
            w = ngram_lm.generate(text_seed=ctx, random_seed=pyrandom.randint(0, 1<<30))
        except Exception:
            break
        if w in ('</s>', None): break
        out.append(w)
    s = ' '.join(out)
    ngram_samples.append({'prompt': ' '.join(p), 'generation': s})
    print(f'[N-GRAM] PROMPT: {" ".join(p)}\n[N-GRAM] GEN   : {s}\n')

# LSTM samples
lstm_samples = []
for prompt in ['the film', 'in the', 'he was']:
    s = lstm_generate(prompt, max_len=20, temperature=0.8)
    lstm_samples.append({'prompt': prompt, 'generation': s})
    print(f'[LSTM]  PROMPT: {prompt}\n[LSTM]  GEN   : {s}\n')

[N-GRAM] PROMPT: the film
[N-GRAM] GEN   : the film starts to fragment " was as if the dock area . most beliefs surrounding the klingons to the polish underground

[N-GRAM] PROMPT: in the
[N-GRAM] GEN   : in the persian emperor then ravaged cappadocia , cilicia and claimed that " behind the font was carved from fine grained white

[N-GRAM] PROMPT: he was
[N-GRAM] GEN   : he was close to patriote which pulled away , making no guarantees as to torture and murder which are covered with tribbles

[LSTM]  PROMPT: the film
[LSTM]  GEN   : the film featured on the city of the 1840s of <unk> , gaining a earthquake of the academic , when although they

[LSTM]  PROMPT: in the
[LSTM]  GEN   : in the main time , tom damage ( reaching <unk> ) , <unk> <unk> , and <unk> of the <unk> having the

[LSTM]  PROMPT: he was
[LSTM]  GEN   : he was among <unk> as a <unk> with <unk> , ( <unk> ) , <unk> , and so <unk> . the 65



In [14]:
summary = pd.DataFrame([
    {'Model': 'Trigram (Kneser-Ney)', 'Perplexity': ngram_ppl, 'Train Time (s)': ngram_train_time},
    {'Model': 'LSTM-LM',              'Perplexity': lstm_ppl,  'Train Time (s)': lstm_train_time},
])
print(summary.to_string(index=False))
summary.to_csv('results/q5_summary.csv', index=False)

with open('results/q5_full.json','w') as f:
    json.dump({
        'ngram': {'perplexity': ngram_ppl, 'train_time_sec': ngram_train_time, 'samples': ngram_samples},
        'lstm':  {'perplexity': lstm_ppl,  'train_time_sec': lstm_train_time,  'samples': lstm_samples},
    }, f, indent=2)
print('\nResults saved to results/')

               Model   Perplexity  Train Time (s)
Trigram (Kneser-Ney) 19820.295237       12.229813
             LSTM-LM   272.068752       39.009538

Results saved to results/


## 6. Discussion (in Report)

See `reports/CENG467_Midterm_Report.pdf` for analysis covering:
- Perplexity differences between count-based and neural models
- Fluency and long-range coherence in generated text
- Vocabulary coverage and OOV handling
- Trade-offs in training cost, memory footprint, and quality